<a href="https://colab.research.google.com/github/Levan-Danelia/FRTB/blob/main/FRTB_DRC_non_ACTP.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd

# Set display options for better readability
pd.set_option('display.float_format', '{:,.0f}'.format)

# ---
# Step 1: Identify Portfolio Positions
# ---
print("Step 1: Initial Portfolio Data")
data = {
    'Bucket': ['Europe_CLO', 'Europe_CLO', 'Europe_CMBS', 'Europe_CMBS'],
    'Position_ID': ['CLO 1', 'CLO 2', 'CMBS 1', 'CMBS 2'],
    'Market_Value': [198000, 163000, 100000, 28000],
    'Risk_Weight_Percent': [1250.0, 1250.0, 1250.0, 1250.0]
}
df = pd.DataFrame(data)
print(df)
print("\n" + "="*50 + "\n")

# ---
# Step 2: Determine Net JTD (Article 325z(1))
# ---
print("Step 2: Determine Net JTD")
print("Per Art. 325z(1), Gross JTD = Market Value.")
print("Assuming no identical tranches, Net JTD = Gross JTD.")
df['Net_JTD'] = df['Market_Value']
print(df[['Position_ID', 'Market_Value', 'Net_JTD']])
print("\n" + "="*50 + "\n")

# ---
# Step 3: Calculate Weighted JTD (Article 325aa(1))
# ---
print("Step 3: Calculate Weighted JTD")
print("Weighted JTD = Net JTD * (0.08 * Risk Weight)")

# Convert percentage to decimal
df['RW_Decimal'] = df['Risk_Weight_Percent'] / 100.0

# Calculate the 8% multiplier
df['Multiplier'] = 0.08 * df['RW_Decimal']

# Calculate the Weighted JTD
df['Weighted_JTD'] = df['Net_JTD'] * df['Multiplier']

print(df[['Position_ID', 'Net_JTD', 'Risk_Weight_Percent', 'Multiplier', 'Weighted_JTD']])
print("\n" + "="*50 + "\n")

# ---
# Step 4: Apply Capping (Article 325aa(3))
# ---
print("Step 4: Apply Capping")
print("Charge is capped at Market Value: min(Weighted JTD, Market Value)")

# The charge is the minimum of the Weighted JTD and the Market Value.
df['Capped_JTD_Charge'] = df[['Weighted_JTD', 'Market_Value']].min(axis=1)

print(df[['Position_ID', 'Market_Value', 'Weighted_JTD', 'Capped_JTD_Charge']])
print("\n" + "="*50 + "\n")


# ---
# Step 5: Assign to Buckets and Aggregate
# ---
print("Step 5: Intra-Bucket Aggregation (Art. 325aa(6))")
# Since all positions are long, this is a simple sum per bucket.
bucket_drc = df.groupby('Bucket')['Capped_JTD_Charge'].sum()
print(bucket_drc)
print("\n" + "="*50 + "\n")

# ---
# Step 6: Final DRC Aggregation
# ---
print("Step 6: Final DRC Aggregation (Art. 325aa(7))")
print("Total DRC is the simple sum of all bucket charges.")
total_drc = bucket_drc.sum()
print(f"\nTotal DRC (non-ACTP) = {total_drc:,.0f}")

Step 1: Initial Portfolio Data
        Bucket Position_ID  Market_Value  Risk_Weight_Percent
0   Europe_CLO       CLO 1        198000                1,250
1   Europe_CLO       CLO 2        163000                1,250
2  Europe_CMBS      CMBS 1        100000                1,250
3  Europe_CMBS      CMBS 2         28000                1,250


Step 2: Determine Net JTD
Per Art. 325z(1), Gross JTD = Market Value.
Assuming no identical tranches, Net JTD = Gross JTD.
  Position_ID  Market_Value  Net_JTD
0       CLO 1        198000   198000
1       CLO 2        163000   163000
2      CMBS 1        100000   100000
3      CMBS 2         28000    28000


Step 3: Calculate Weighted JTD
Weighted JTD = Net JTD * (0.08 * Risk Weight)
  Position_ID  Net_JTD  Risk_Weight_Percent  Multiplier  Weighted_JTD
0       CLO 1   198000                1,250           1       198,000
1       CLO 2   163000                1,250           1       163,000
2      CMBS 1   100000                1,250           1     